# 📈 JP Backtest — Complete System Architecture Walkthrough

Welcome to the **Interactive Architecture Guide** for the Japan Stock Backtesting Application. 

This single notebook will walk you linearly through the entire lifecycle of a daily stock price from the exact moment it is pulled from the stock market to the exact moment it visually renders on the dashboard for the end-user.

You won't just read about the architecture—**you will actively run the code that powers it**. We will simulate the heavy lifting of the message queue, the horizontal scaling of Apache Spark, and the real-time inference of the AWS Machine Learning model.

***Note:*** *You can actively run the code cells below by clicking on them and pressing `Shift + Enter`.*

In [ ]:
# Run this cell first to ensure all required dependencies for the simulation are installed in your environment.
!pip install yfinance pandas numpy quantstats xgboost pyspark scikit-learn

---
## Phase 1: The Producer (Data Ingestion via `yfinance`)

The journey begins at the edge of our system. A Python microservice acts as the **Producer**. Its only job is to constantly poll the Yahoo Finance API for the latest daily Open, High, Low, Close, and Volume (OHLCV) records for Japanese blue-chip stocks.

Let's simulate the Producer actively pulling a live tick.

In [ ]:
import yfinance as yf
import pandas as pd

ticker = '7203.T'  # Toyota Motor Corp
print(f"📡 PRODUCER: Fetching live data for {ticker}...")

# Fetch the last 10 days of active market data
history = yf.download(ticker, period='10d', progress=False)

if isinstance(history.columns, pd.MultiIndex):
    history.columns = history.columns.droplevel(1)

print("✅ PRODUCER: Successfully grabbed active market ticks:")
display(history[['Open', 'High', 'Low', 'Close', 'Volume']].tail(3))

---
## Phase 2: The Message Queue (BlazingMQ & The 'Slow Consumer' Problem)

Now that the Producer has the data, it needs to save it to our database. 

**Why not just save it directly to the database?**
If the market opens and thousands of ticker price updates flow in per second, writing them to a massive Delta Lake/Database requires network calls, data validation, and heavy disk I/O. If the database gets slow, your Producer (the scraper) gets stuck waiting, causing it to crash or miss moving prices.

To solve this, we place **BlazingMQ** (Bloomberg's high-performance open-source message queue) between the Producer and the Consumer (Database Writer). 

### 👨‍💻 Experience the Actual BlazingMQ Flow
If your BlazingMQ backend is running in the background, the code below will allow you to session-post the actual data we pulled in Phase 1 into the real broker queue. 

**Note:** We have included a robust 'Mock Fallback' so if your broker is not currently running, the notebook will elegantly simulate the success while showing you the exact production code syntax used in Bloomberg environments.

In [ ]:
import json
import time
from datetime import datetime

# 1. Connection settings from our app config
BMQ_BROKER_URI = "tcp://localhost:30114"
BMQ_QUEUE_URI = "bmq://bmq.test.mmap.priority/stock-data-queue"

print(f"🔗 Initializing BlazingMQ Session to {BMQ_BROKER_URI}...")

try:
    import blazingmq
    # Production Mode
    session = blazingmq.Session(blazingmq.session_options.SessionOptions(broker_uri=BMQ_BROKER_URI))
    session.open(BMQ_QUEUE_URI, options=blazingmq.QueueOptions(), write=True)
    CONNECTED = True
    print("✅ CONNECTED: Successfully established session following Bloomberg SDK patterns.")
except (ImportError, Exception) as e:
    # Mock Simulation Mode if library/broker is missing
    CONNECTED = False
    print(f"⚠️ MOCK MODE: (Reason: {e})")
    print(f"👉 In production, we would now be posting to {BMQ_QUEUE_URI}")

# 2. Prepare the actual message from our live yfinance data
last_close = history['Close'].iloc[-1]
payload = {
    "ticker": "7203.T",
    "price": float(last_close),
    "timestamp": datetime.now().isoformat(),
    "metadata": {"source": "jupyter_notebook_manual_check", "accuracy": "real-time"}
}

print(f"\n📤 Sending payload: {json.dumps(payload, indent=2)}")

if CONNECTED:
    # In real BMQ, we post individual message payloads onto the URI
    session.post(BMQ_QUEUE_URI, json.dumps(payload).encode('utf-8'))
    print("\n✅ SUCCESS: Message pushed to the live BlazingMQ cluster!")
    session.close(BMQ_QUEUE_URI)
    session.stop()
else:
    print("\n✅ MOCK SUCCESS: Simulation of the asynchronous POST call is complete.")

### Why this works so well:
Notice that the notebook didn't have to wait for the database, Databricks, or anything else to "respond". Once the `.post()` call is made to the BlazingMQ broker, our Producer script is immediately free to go fetch the next symbol. 

Let's visualize the **Buffer Logic** below with a multi-threaded simulation to see the speed difference in action:

In [ ]:
import queue
import threading

# Simulating 5 rapid data ticks fetched from the live market
market_ticks = [2800.0, 2801.5, 2802.0, 2799.0, 2805.0]

blazing_mq_buffer = queue.Queue()  # Simulating the internal Broker memory

def simulator_producer():
    start_time = time.time()
    for price in market_ticks:
        blazing_mq_buffer.put(price) # This represents the session.post() above
        print(f"[{time.strftime('%H:%M:%S')}] 🚀 PRODUCER: Instantly queued payload: {price}")
        time.sleep(0.01) 
        
    print(f"✅ PRODUCER: Finished all my work in {time.time() - start_time:.2f} seconds!")

def simulator_consumer():
    while not blazing_mq_buffer.empty():
        price = blazing_mq_buffer.get()
        # Simulates a heavy, slow database insertion (like saving to Delta Lake)
        time.sleep(1.0) 
        print(f"[{time.strftime('%H:%M:%S')}] 💾 CONSUMER: Slowly, but safely saved payload to Database: {price}")
        blazing_mq_buffer.task_done()
    print("✅ CONSUMER: Automatically finished draining the queue!")

print("--- Starting Decoupled MQ Integration Simulation ---")
p_thread = threading.Thread(target=simulator_producer)
c_thread = threading.Thread(target=simulator_consumer)

p_thread.start()
time.sleep(0.1)
c_thread.start()

p_thread.join()
c_thread.join()

---
## Phase 3: The Data Platform (Databricks, Apache Spark, & Delta Lake)

When the Consumer drains BlazingMQ, it saves the payloads permanently into a **Databricks Delta Lake**.

**Why Databricks instead of PostgreSQL?**
Financial Feature Engineering requires running intense math (like a 200-day rolling average) across millions of rows spanning thousands of stocks. A standard relational database scales *vertically* (one single massive computer hard-drive), meaning it will eventually freeze on huge queries. Databricks orchestrates **Apache Spark**, which scales *horizontally*, splitting the math dataset into a thousand smaller chunks across hundreds of separate computers simultaneously in RAM.

Let's spin up a local PySpark cluster right now to simulate distributed feature engineering:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg

print("Spinning up local Apache Spark cluster...")
spark = SparkSession.builder.appName("TradingBacktestApp").getOrCreate()

spark_df = spark.createDataFrame(
    history.reset_index()[['Date', 'Close']].assign(Ticker='7203.T')
)

# The Window explicitly groups the math by Ticker, so Spark knows exactly how to distribute the load across multiple computers
windowSpec = Window.partitionBy("Ticker").orderBy("Date").rowsBetween(-4, 0) # 5-day Simple Moving Average

# Calculate the average concurrently across the distributed window
df_features = spark_df.withColumn("SMA_5", avg(col("Close")).over(windowSpec))

print("\nDistributed Apache Spark Feature Engineering Complete:")
df_features.show(5)

---
## Phase 4: Machine Learning (AWS SageMaker)

The clean "feature data" calculated by Spark (SMA, RSI) forms the ground truth for our active trading application. We now pipe this data securely out of Databricks and straight into **AWS SageMaker**.

SageMaker hosts our **XGBoost Classifier Model**. 
- In **Training Mode**, we feed it 10 years of feature data to find complex, non-linear correlations predicting stock drift.
- In **Inference Mode**, we pass it today's newly calculated Databricks row, and it instantly responds with a highly-confident `BUY`, `SELL`, or `HOLD` flag.

Let's actively simulate training an algorithmic trading classifier right now:

In [ ]:
import xgboost as xgb
import numpy as np
from sklearn.metrics import accuracy_score

# 1. Synthesize 1000 days of Feature Data for Training (RSI, SMA, Momentum)
print("🧠 AWS SageMaker: Piping historical Delta Lake feature data into modeling engine...")
np.random.seed(42)
X_train = np.random.rand(1000, 3) # 3 Features

# Target: '1' if the stock will go up 5% in the next 20 days, '0' otherwise.
y_train = (X_train[:, 0] + X_train[:, 1] > 1.2).astype(int) 
X_test = np.random.rand(200, 3)
y_test = (X_test[:, 0] + X_test[:, 1] > 1.2).astype(int)

# 2. Train the XGBoost Backtesting Model
model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(f"\n✅ AWS SageMaker: Training complete! Simulation Accuracy: {accuracy_score(y_test, preds):.2%}")

print("\n🔮 Inference: If we pass today's live stock data to the SageMaker InvokeEndpoint API...")
live_prediction = model.predict(np.array([[0.8, 0.9, 0.2]]))
decision = "BUY" if live_prediction[0] == 1 else "HOLD/SELL"
print(f"➡️ SageMaker Responds: {decision}")

---
## Phase 5: The Middleware (Java Spring Boot API)

With the Data perfected by **Databricks** and the Intelligence generated by **AWS SageMaker**, we need to safely expose this state to the open internet. We do not let Web/Mobile apps query our data lake or models directly!

Our **Java Spring Boot API** acts as the secure middleman block of the application. It fetches the aggregated state, caches it in memory, handles enterprise security routing (CORS/Authentication), and serves strictly-typed JSON through REST standard endpoints (`@GetMapping("/api/stocks")`).

---
## Phase 6: The Frontend (React + Vite SPA)

The final stop is the user's browser. Our dynamic **React 18** frontend fetches the flawless JSON output produced by the Java backend using `axios`.

Because all the heavy-lifting (the yFinance polling, the Spark distributed window math, the XGBoost Decision Trees) occurred upstream, the React frontend is extraordinarily fast. It simply Maps the JSON array into beautifully themed CSS tracking cards, attaching native CSS `onHover` tooltip layers for user explanation.

### Complete Flow Conclusion:
**`yFinance Trigger ➔ BlazingMQ Buffer ➔ Spark Feature Engineering ➔ SageMaker AI Inference ➔ Java Spring Boot Security ➔ React Dashboard Rendering.`**